# 用 LightGBM 与 XGBoost 做零售需求预测

读取与 Prophet 教程相同的商超、红酒、庭院、奢侈品手袋四套 CSV，输出训练内回看、四个历史滚动起点与最终测试成绩。最后 28 天为 2025-12-04 至 2025-12-31。

从仓库根目录启动 Jupyter，并先安装 `pyproject.toml`，然后 Restart Kernel and Run All。`src/retail_forecast/common.py` 保存共同数据协议，各模型模块保存可检查的完整训练实现。所有输入 CSV 不改写。

训练内回看只描述拟合能力；历史窗口按业务生成机制预先指定，每个起点仅使用当时历史。评价排除闭店/缺货标签，所有方案共用日期和非负/营业日历处理。商超与红酒促销全年重复，因此促销观察窗口仍混合促销和非促销日，另外输出条件切片。

奢侈品约 94% 的营业日销量为零，窗口内没有销量时 WAPE 无定义（记为缺失），需结合 MAE 与 RMSE 解读。

## 1. 数据、业务任务与评估协议

四个案例分别代表平滑加法需求（商超）、低销量计数需求（红酒）、强季节需求（庭院）和间歇性高价值需求（奢侈品手袋：先决定是否发生需求，再抽取 1/2/5/10 件）。诊断真值 `conditional_mean`、`latent_demand` 不进入特征。

奢侈品只有周六/周日、12 月旺季、VIP 预览和折扣这些业务输入。树模型的早停依赖内部验证窗口，而该窗口几乎没有销量，选出的树数量会很少，需要结合 `selected_rounds` 解读。

In [ ]:
# 首次使用请在仓库根目录运行：uv sync --group dev
from pathlib import Path
import os, sys


def find_repo_root():
    configured = os.environ.get("RETAIL_FORECAST_ROOT")
    candidates = (
        [Path(configured).expanduser()]
        if configured
        else [Path.cwd(), *Path.cwd().parents]
    )
    for candidate in candidates:
        candidate = candidate.resolve()
        if (candidate / "pyproject.toml").is_file() and (
            candidate / "notebooks"
        ).is_dir():
            return candidate
    raise RuntimeError(
        "找不到仓库根目录；请从仓库内启动 Jupyter，或设置 RETAIL_FORECAST_ROOT"
    )


REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT / "src"))
import inspect, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Code
from retail_forecast.common import *

assert list(SCENARIOS) == ["grocery", "wine", "garden", "luxury"]
display(
    pd.DataFrame(
        [
            {
                "scenario": n,
                "days": len(f),
                "cutoff": cutoff,
                "stockout_days": int(f.stockout.sum()),
                "open_mean_sales": float(f.loc[f.is_open, "y"].mean()),
                "open_zero_rate": float(f.loc[f.is_open, "y"].eq(0).mean()),
            }
            for n, f in datasets.items()
        ]
    )
)
display(pd.DataFrame([{"scenario": n, **w} for n in SCENARIOS for w in windows(n)]))

## 2. 梯度提升与多步特征

$$\hat y_i^{(m)}=\hat y_i^{(m-1)}+\eta f_m(\mathbf x_i).$$

变量：$i$ 是样本编号，$m$ 是提升轮数，$\eta$ 是学习率，$f_m$ 是第 $m$ 棵回归树，$\mathbf x_i$ 是特征向量。新树学习损失函数要求的修正方向，平方误差下与残差有关。

预测起点特征将截至起点的最近销量、前七天位置销量、7/28 天均值与有效比例冻结，再拼上未来日期的已知特征及 horizon。整个 28 天内这些历史摘要固定，不读取未来真实销量。训练历史按每 7 天一个起点展开未来 28 天标签；同一日期可作为不同起点的训练标签，样本存在重叠。

比较 LightGBM/XGBoost 的日历业务版本与 `+ history` 版本，以及 Seasonal naive。该消融同时改变了特征与训练样本组织，不能把差异全部归因于单个滞后特征。


In [ ]:
scenario = "grocery"
X = make_features(datasets[scenario], scenario)
display(X.head())
print("Feature columns:", list(X))
print("Truth excluded:", not {"y", "latent_demand", "conditional_mean"}.intersection(X))

## 3. 训练实现与参数检查

完整实现可在配套 Python 模块阅读。下面直接展示当前运行的训练函数，保证解释与代码一致。

In [ ]:
from retail_forecast.trees import *

display(Code(inspect.getsource(fit_tree), language="python"))
display(Code(inspect.getsource(origin_features), language="python"))

## 4. 实际训练、滚动验证与最终测试

每个历史窗口重新拟合；TFT 默认恢复此前在同一起点训练的检查点。树和神经网络的内部验证全部位于该起点之前。最终模型的历史回看参数已见过训练标签。

In [ ]:
m, p, details, summary, finals = run_family("trees", NAMES, fit_tree, predict_tree)
display(m.round(4))
display(summary.round(4))
display(details)

## 5. 预测与性能图

训练、滚动验证和最终测试来自不同时间段。比较 WAPE 时同时考虑销量分母，并查看 MAE、RMSE 和偏差。

In [ ]:
colors = plot_family("trees", NAMES, m, p, summary)

## 6. 参数与诊断

诊断用于发现结构遗漏，未来泛化由时间窗口评价。特征重要性与注意力都需要业务解释，不能直接作为因果效应。

In [ ]:
importance, holiday_support = explain_trees(finals)
display(importance.head(20))
display(holiday_support)

## 7. 结果口径与系统交付

输出包括预测起点、目标日期、步长、原始预测和营业日历处理后的销售预测。补货系统还要结合提前期、库存和服务水平。当前脚本提供本地可重复实验；生产发布需补齐数据快照、权限、调度、质量门槛与回滚。

In [ ]:
out = RUNS_DIR / "trees"
assert len(m) == len(SCENARIOS) * len(NAMES) * 6
assert m.loc[m.split.eq("test"), "n"].nunique() == 1
for name, digest in INPUT_HASHES.items():
    assert hashlib.sha256((DATA_DIR / name).read_bytes()).hexdigest() == digest
print("Exports:", out.resolve())
display(summary.pivot(index="scenario", columns="model", values="WAPE"))